# HySpecNet Patch Bitstream Sizes

This Colab notebook measures actual per-sample bitstream sizes for three HySpecNet-11k `easy/test` patches used in thesis reconstruction examples. It runs the real `compress/decompress` path for `hierarchical_spectral_mamba_ae` K=4 + spatial, RD lambda 0.0003.

The notebook does not edit thesis files or presentation files. It writes a small report artifact to Google Drive and prints the requested Markdown table.

## 1. Settings

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/mhx1467/master-thesis-code.git'
REPO_DIR = Path('/content/hsi')
REPO_REF = 'main'

DRIVE_HSI = Path('/content/drive/MyDrive/hsi')
HYSPECNET_ARCHIVE = DRIVE_HSI / 'data/archives/hyspecnet_easy_test_data_npy_2026-06-13.tar.zst'
HYSPECNET_ARCHIVE_SHA256 = DRIVE_HSI / 'data/archives/hyspecnet_easy_test_data_npy_2026-06-13.tar.zst.sha256'
HYSPECNET_EXTRACT_PARENT = Path('/content/hsi_data')
HYSPECNET_ROOT = HYSPECNET_EXTRACT_PARENT / 'hyspecnet_easy_test_data_npy'
HYSPECNET_SPLIT = HYSPECNET_ROOT / 'splits/easy/test.csv'
EXPECTED_HYSPECNET_FILES = 1149

# First candidate matches the thesis prompt filename. The second matches the local canonical name.
CHECKPOINT_CANDIDATES = [
    DRIVE_HSI / 'checkpoints/hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_01_best.pt',
    DRIVE_HSI / 'checkpoints/hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_0003_best.pt',
]

ARTIFACT_ROOT = DRIVE_HSI / 'remote_artifacts/hyspecnet_patch_bitstreams_2026-06-15'

SAMPLES = [
    {
        'index': 29,
        'patch_id': 'ENMAP01-____L2A-DT0000004981_20221102T031508Z_003_V010110_20221116T141823Z-Y03990526_X09431070',
    },
    {
        'index': 43,
        'patch_id': 'ENMAP01-____L2A-DT0000004981_20221102T031526Z_007_V010110_20221118T204141Z-Y03990526_X09421069',
    },
    {
        'index': 28,
        'patch_id': 'ENMAP01-____L2A-DT0000004981_20221102T031508Z_003_V010110_20221116T141823Z-Y02710398_X05590686',
    },
]

EXPECTED_SHAPE_CHW = (202, 128, 128)
EXPECTED_VALUES = 202 * 128 * 128
ORIGINAL_BITS_PER_CHANNEL = 16.0

REQUIRE_CUDA = True
FORCE_REINSTALL_ENV = False

print('HySpecNet archive:', HYSPECNET_ARCHIVE)
print('HySpecNet root:', HYSPECNET_ROOT)
print('Checkpoint candidates:')
for path in CHECKPOINT_CANDIDATES:
    print(' -', path)
print('Artifact root:', ARTIFACT_ROOT)

## 2. Mount Drive and Prepare Repository

In [ ]:
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)

for module_name in list(sys.modules):
    if module_name == 'hsi_compression' or module_name.startswith('hsi_compression.'):
        del sys.modules[module_name]

os.chdir(REPO_DIR)
print('Repo:', Path.cwd())
print('Git:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

## 3. Install Runtime Dependencies

This uses the same prebuilt `causal-conv1d` and `mamba-ssm` wheel pattern as the other Colab notebooks. On the first run, the cell intentionally restarts the runtime after installing binary modules. After reconnecting, rerun from the settings cell.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PIP = [sys.executable, '-m', 'pip']
ENV_MARKER = Path('/content/.hsi_compression_patch_bitstreams_env_v1_torch27_mamba232')


def run(cmd, *, required=True):
    print('Running:', ' '.join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout[-6000:])
    if result.returncode != 0:
        message = 'Command failed with exit code %s: %s' % (
            result.returncode,
            ' '.join(map(str, cmd)),
        )
        if required:
            raise RuntimeError(message)
        print('Optional command failed:', message)
        return False
    return True


if FORCE_REINSTALL_ENV and ENV_MARKER.exists():
    ENV_MARKER.unlink()

run(['apt-get', 'update'], required=False)
run(['apt-get', 'install', '-y', 'zstd'], required=False)

if not ENV_MARKER.exists():
    run(PIP + ['install', '-q', '--upgrade', 'pip', 'setuptools<82', 'wheel', 'packaging', 'pybind11', 'ninja'])
    run(PIP + [
        'install', '-q', '--force-reinstall',
        'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ])
    run(PIP + [
        'install', '-q', '--upgrade', '--force-reinstall',
        'numpy==1.26.4', 'pandas==2.2.2', 'scipy>=1.12,<1.15', 'scikit-learn>=1.6,<1.8',
    ])
    run(PIP + ['install', '-q', '-e', '.', 'tqdm', 'matplotlib', 'ipywidgets', 'tabulate'])

    import torch
    cxx11_abi = 'TRUE' if getattr(torch._C, '_GLIBCXX_USE_CXX11_ABI', True) else 'FALSE'
    python_tag = 'cp%d%d' % (sys.version_info.major, sys.version_info.minor)
    if python_tag != 'cp312':
        raise RuntimeError('This prebuilt Mamba preset expects Python 3.12, got %s.' % python_tag)
    if cxx11_abi != 'TRUE':
        raise RuntimeError('This prebuilt Mamba preset expects Torch CXX11 ABI TRUE, got %s.' % cxx11_abi)
    print('Torch CXX11 ABI:', cxx11_abi)

    run(PIP + [
        'install', '-q', '--force-reinstall', '--no-deps',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    ])
    run(PIP + [
        'install', '-q', '--force-reinstall', '--no-deps',
        'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    ])
    ENV_MARKER.write_text('installed\n', encoding='utf-8')
    print('Dependencies installed. Restarting runtime to reload binary modules.')
    os.kill(os.getpid(), 9)
else:
    print('Dependency marker exists, skipping reinstall:', ENV_MARKER)

import numpy as np
import torch
print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required. In Colab choose Runtime -> Change runtime type -> GPU.')
if not torch.__version__.startswith('2.7.'):
    if ENV_MARKER.exists():
        ENV_MARKER.unlink()
    raise RuntimeError(
        'Mamba prebuilt wheels require Torch 2.7.x, but active Torch is %s. Restart the Colab runtime and rerun from the settings cell.' % torch.__version__
    )
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    subprocess.run(['nvidia-smi'], check=False)

try:
    from mamba_ssm import Mamba  # noqa: F401
    print('mamba-ssm import: ok')
except Exception as exc:
    raise RuntimeError(
        'mamba-ssm is required for this notebook. Use a fresh GPU Colab runtime, set FORCE_REINSTALL_ENV=True, and rerun from the repo cell.'
    ) from exc

## 4. Prepare Data and Checkpoint

In [ ]:
import csv
import hashlib
import json
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def count_hyspecnet_files(root: Path) -> int:
    patches = root / 'patches'
    return sum(1 for _ in patches.rglob('*-DATA.npy')) if patches.exists() else 0


def sha256_file(path: Path, chunk_size: int = 32 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def verify_archive_checksum() -> str | None:
    if not HYSPECNET_ARCHIVE_SHA256.exists():
        print('Checksum file missing; skipping archive SHA-256 verification:', HYSPECNET_ARCHIVE_SHA256)
        return None
    expected = HYSPECNET_ARCHIVE_SHA256.read_text(encoding='utf-8').split()[0]
    actual = sha256_file(HYSPECNET_ARCHIVE)
    if actual != expected:
        raise RuntimeError('Archive checksum mismatch: expected %s, got %s' % (expected, actual))
    print('Archive SHA-256 verified:', actual)
    return actual


def ensure_hyspecnet_data() -> None:
    existing = count_hyspecnet_files(HYSPECNET_ROOT)
    if existing == EXPECTED_HYSPECNET_FILES and HYSPECNET_SPLIT.exists():
        print('HySpecNet DATA.npy subset already extracted:', HYSPECNET_ROOT, existing, 'files')
        return
    if not HYSPECNET_ARCHIVE.exists():
        raise FileNotFoundError(
            'Missing HySpecNet archive: %s. Upload/extract hyspecnet_easy_test_data_npy or adjust HYSPECNET_ROOT.' % HYSPECNET_ARCHIVE
        )
    HYSPECNET_EXTRACT_PARENT.mkdir(parents=True, exist_ok=True)
    verify_archive_checksum()
    if HYSPECNET_ROOT.exists():
        shutil.rmtree(HYSPECNET_ROOT)
    print('Extracting HySpecNet archive to:', HYSPECNET_EXTRACT_PARENT)
    subprocess.run(
        ['tar', '-I', 'zstd', '-xf', str(HYSPECNET_ARCHIVE), '-C', str(HYSPECNET_EXTRACT_PARENT)],
        check=True,
    )
    extracted = count_hyspecnet_files(HYSPECNET_ROOT)
    if extracted != EXPECTED_HYSPECNET_FILES:
        raise RuntimeError('Expected %d DATA.npy files, found %d' % (EXPECTED_HYSPECNET_FILES, extracted))
    if not HYSPECNET_SPLIT.exists():
        raise FileNotFoundError('Missing split file after extraction: %s' % HYSPECNET_SPLIT)
    print('Extracted HySpecNet DATA.npy subset:', extracted, 'files')


def select_checkpoint() -> Path:
    for path in CHECKPOINT_CANDIDATES:
        if path.exists():
            return path
    raise FileNotFoundError('No checkpoint found. Checked: %s' % CHECKPOINT_CANDIDATES)


ensure_hyspecnet_data()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = select_checkpoint()

with HYSPECNET_SPLIT.open(newline='', encoding='utf-8') as f:
    split_entries = [row[0].strip() for row in csv.reader(f) if row and row[0].strip()]

print('Split:', HYSPECNET_SPLIT, 'rows:', len(split_entries))
for sample in SAMPLES:
    entry = split_entries[sample['index']]
    if sample['patch_id'] not in entry:
        raise RuntimeError('Split index %d mismatch: %s' % (sample['index'], entry))
    path = HYSPECNET_ROOT / 'patches' / entry
    if not path.exists():
        raise FileNotFoundError(path)
    sample['split_entry'] = entry
    sample['data_npy_path'] = str(path)
    print(sample['index'], sample['patch_id'], path.stat().st_size, 'bytes')

print('Checkpoint:', CHECKPOINT_PATH)
print('Checkpoint SHA-256:', sha256_file(CHECKPOINT_PATH))

## 5. Measure Per-Patch Bitstreams

In [ ]:
import json
import math
import os
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Markdown, display

from hsi_compression.datasets import HSITiffDataset
from hsi_compression.engine.checkpointing import load_checkpoint
from hsi_compression.metrics import (
    compute_actual_bpppc_from_strings,
    compute_compression_ratio_from_bpppc,
    psnr,
    ref_sam_deg,
)
from hsi_compression.models.registry import build_model


def sum_string_bytes(obj) -> int:
    if isinstance(obj, (bytes, bytearray)):
        return len(obj)
    if isinstance(obj, (list, tuple)):
        return sum(sum_string_bytes(item) for item in obj)
    raise TypeError('Unsupported strings container type: %r' % (type(obj),))


def mib(num_bytes: int | float) -> float:
    return float(num_bytes) / (1024.0 ** 2)


def kib(num_bytes: int | float) -> float:
    return float(num_bytes) / 1024.0


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if REQUIRE_CUDA and device.type != 'cuda':
    raise RuntimeError('GPU runtime is required for this Mamba measurement.')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
config = ckpt['config']
model_section = config['model']
model_kwargs = dict(model_section['model_kwargs'])
model_name = model_section['model_name']
rd_lambda = config.get('training', {}).get('rd_lambda')

model = build_model(
    model_name=model_name,
    in_channels=model_kwargs.get('in_channels', 202),
    **{k: v for k, v in model_kwargs.items() if k != 'in_channels'},
).to(device)
load_checkpoint(CHECKPOINT_PATH, model=model, optimizer=None, map_location=device)
model.eval()
if hasattr(model, 'update'):
    model.update(force=True)

print('Device:', device)
print('Model:', model_name)
print('Checkpoint rd_lambda:', rd_lambda)

paths = [Path(sample['data_npy_path']) for sample in SAMPLES]
dataset = HSITiffDataset(paths, return_mask=True, prefer_npy=True, npy_mmap=False)

records = []
with torch.no_grad():
    for local_idx, sample_spec in enumerate(SAMPLES):
        sample = dataset[local_idx]
        path = Path(sample['path'])
        x_chw = sample['x'].contiguous()
        mask_chw = sample['valid_mask'].contiguous()
        shape_chw = tuple(int(v) for v in x_chw.shape)
        if shape_chw != EXPECTED_SHAPE_CHW:
            raise RuntimeError('Unexpected tensor shape for %s: %s' % (path, shape_chw))
        values = int(math.prod(shape_chw))
        if values != EXPECTED_VALUES:
            raise RuntimeError('Unexpected value count: %s' % values)

        x = x_chw.unsqueeze(0).to(device=device, dtype=torch.float32)
        mask = mask_chw.unsqueeze(0).to(device=device)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        encode_start = time.perf_counter()
        packed = model.compress(x, valid_mask=mask)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        encode_sec = time.perf_counter() - encode_start

        compressed_bytes = sum_string_bytes(packed['strings'])
        actual_bpppc = compute_actual_bpppc_from_strings(packed['strings'], tuple(x.shape))
        actual_bpppc_direct = compressed_bytes * 8.0 / values
        if abs(actual_bpppc - actual_bpppc_direct) > 1e-12:
            raise RuntimeError('bpppc mismatch: repo=%r direct=%r' % (actual_bpppc, actual_bpppc_direct))

        if device.type == 'cuda':
            torch.cuda.synchronize()
        decode_start = time.perf_counter()
        decoded = model.decompress(
            strings=packed['strings'],
            shape=packed['shape'],
            z_shape=packed.get('z_shape'),
        )
        if device.type == 'cuda':
            torch.cuda.synchronize()
        decode_sec = time.perf_counter() - decode_start

        if not isinstance(decoded, dict) or 'x_hat' not in decoded:
            raise RuntimeError('model.decompress() did not return a dict with x_hat')
        x_hat = decoded['x_hat'].to(device=device, dtype=torch.float32)
        decompress_ok = tuple(x_hat.shape) == tuple(x.shape) and bool(torch.isfinite(x_hat).all().item())
        psnr_db = float(psnr(x_hat, x, data_range=1.0).item())
        sam_deg = float(ref_sam_deg(x_hat, x).item())

        record = {
            'index': int(sample_spec['index']),
            'patch_id': str(sample['patch_id']),
            'data_npy_path': str(path),
            'shape': list(shape_chw),
            'values': values,
            'raw_16bit_bytes': values * 2,
            'raw_16bit_mib': mib(values * 2),
            'data_npy_bytes': int(path.stat().st_size),
            'data_npy_mib': mib(path.stat().st_size),
            'compressed_bytes': int(compressed_bytes),
            'compressed_kib': kib(compressed_bytes),
            'actual_bpppc': float(actual_bpppc),
            'actual_cr': float(compute_compression_ratio_from_bpppc(actual_bpppc, ORIGINAL_BITS_PER_CHANNEL)),
            'psnr_db': psnr_db,
            'sam_deg': sam_deg,
            'decompress_ok': bool(decompress_ok),
            'packed_keys': sorted(packed.keys()),
            'packed_shape': list(packed['shape']) if isinstance(packed.get('shape'), tuple) else packed.get('shape'),
            'packed_z_shape': list(packed.get('z_shape')) if isinstance(packed.get('z_shape'), tuple) else packed.get('z_shape'),
            'encode_sec': encode_sec,
            'decode_sec': decode_sec,
        }
        records.append(record)
        print(json.dumps(record, indent=2))

summary = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'repo_ref': REPO_REF,
    'repo_head': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'checkpoint_path': str(CHECKPOINT_PATH),
    'checkpoint_sha256': sha256_file(CHECKPOINT_PATH),
    'checkpoint_rd_lambda': rd_lambda,
    'data_root': str(HYSPECNET_ROOT),
    'split_csv': str(HYSPECNET_SPLIT),
    'counting_rule': 'compressed_bytes is the recursive sum of byte lengths in packed["strings"]. shape/z_shape are API side information and are not serialized by this model, so no extra metadata bytes are included.',
    'results': records,
}

table_df = pd.DataFrame([
    {
        'index': r['index'],
        'patch_id': r['patch_id'],
        'raw 16-bit MiB': r['raw_16bit_mib'],
        'DATA.npy MiB': r['data_npy_mib'],
        'compressed KiB': r['compressed_kib'],
        'actual_bpppc': r['actual_bpppc'],
        'actual CR': r['actual_cr'],
        'PSNR': r['psnr_db'],
        'SAM': r['sam_deg'],
    }
    for r in records
])

display(table_df)
markdown_table = table_df.to_markdown(index=False, floatfmt='.6f')
print(markdown_table)
display(Markdown(markdown_table))

details = pd.DataFrame([
    {
        'index': r['index'],
        'patch_id': r['patch_id'],
        'DATA.npy path': r['data_npy_path'],
        'shape': str(tuple(r['shape'])),
        'values': r['values'],
        'raw_16bit_bytes': r['raw_16bit_bytes'],
        'DATA.npy bytes': r['data_npy_bytes'],
        'compressed bytes': r['compressed_bytes'],
        'decompress_ok': r['decompress_ok'],
    }
    for r in records
])
display(details)

bpppc_values = [r['actual_bpppc'] for r in records]
compressed_values = [r['compressed_bytes'] for r in records]
spread_bpppc = max(bpppc_values) - min(bpppc_values)
spread_pct = spread_bpppc / (sum(bpppc_values) / len(bpppc_values)) * 100.0
interpretation = [
    'Interpretation:',
    '- Per-sample compressed bitstream sizes differ by %.2f KiB across these three patches (%.2f%% bpppc spread around their mean).' % ((max(compressed_values) - min(compressed_values)) / 1024.0, spread_pct),
    '- These numbers are safe to use as per-sample sizes only if described as measured for these exact three patches and this exact checkpoint/runtime path.',
    '- For benchmark-level RD claims, keep reporting the full easy/test mean actual_bpppc rather than replacing it with these three examples.',
    '- Metadata note: no serialized metadata overhead was included; this model returns shape/z_shape as Python side information for the API, while actual_bpppc in the repo counts packed strings bytes.',
]
print('\n'.join(interpretation))

json_path = ARTIFACT_ROOT / 'hyspecnet_patch_bitstream_sizes.json'
csv_path = ARTIFACT_ROOT / 'hyspecnet_patch_bitstream_sizes.csv'
md_path = ARTIFACT_ROOT / 'hyspecnet_patch_bitstream_sizes.md'
json_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
pd.DataFrame(records).to_csv(csv_path, index=False)
md_path.write_text(markdown_table + '\n\n' + '\n'.join(interpretation) + '\n', encoding='utf-8')
print('Saved:', json_path)
print('Saved:', csv_path)
print('Saved:', md_path)